# Stage 20 Final Held-Out Test Evaluation

This notebook is the guarded final test-set evaluation workflow. Phase 1 prepares and freezes the candidate registry from validation artifacts only. Phase 2 should be run once, after Stage 19 has finished and the registry is frozen. The held-out test split is not evaluated unless `RUN_FINAL_TEST_EVALUATION` is explicitly set to `True`.

In [1]:
from pathlib import Path
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.final_evaluation import (
    DEFAULT_STAGE20_OUTPUT_DIR,
    FINAL_TEST_GUARD_MESSAGE,
    assert_final_registry_ready,
    build_final_candidate_registry,
    freeze_candidate_registry,
    load_prediction_tables,
    materialize_final_candidate_predictions,
    require_final_test_confirmation,
    run_final_test_evaluation,
)

stage20_output_dir = repo_root / DEFAULT_STAGE20_OUTPUT_DIR
prediction_dir = stage20_output_dir / "predictions"
stage20_output_dir.mkdir(parents=True, exist_ok=True)
prediction_dir.mkdir(parents=True, exist_ok=True)

{
    "repo_root": str(repo_root),
    "stage20_output_dir": str(stage20_output_dir),
    "prediction_dir": str(prediction_dir),
}


{'repo_root': '/home/manns79/dreamt-wearable-sleep-staging',
 'stage20_output_dir': '/home/manns79/dreamt-wearable-sleep-staging/results/stage20_final_test_evaluation',
 'prediction_dir': '/home/manns79/dreamt-wearable-sleep-staging/results/stage20_final_test_evaluation/predictions'}

## Candidate Registry

The registry is built from validation artifacts only. Stage 19 will remain pending until its summary exists, and every candidate must be `ready` before the final test run is allowed.

In [2]:
registry = build_final_candidate_registry(results_dir=repo_root / "results")
registry.to_csv(stage20_output_dir / "candidate_registry_draft.csv", index=False)
display(registry)

pending = registry[registry["status"] != "ready"]
if pending.empty:
    print("All candidates are ready to freeze.")
else:
    print("Pending candidates:")
    display(pending[["candidate_id", "status", "notes"]])


,candidate_id,stage,model_class,model_family,model_name,selection_rule,validation_macro_f1,status,validation_artifact_path,test_artifact_path,notes
0,stage6_majority_class,stage6,majority_class,sanity_baseline,majority_class,included representative Stage 6 model class; h...,0.275983,ready,/home/manns79/dreamt-wearable-sleep-staging/re...,None,
1,stage6_logistic_regression,stage6,logistic_elasticnet,feature_baseline,logistic_elasticnet,included representative Stage 6 model class; h...,0.374741,ready,/home/manns79/dreamt-wearable-sleep-staging/re...,None,
2,stage6_xgboost,stage6,xgboost_all_features,feature_baseline,xgboost_all_features,included representative Stage 6 model class; h...,0.383188,ready,/home/manns79/dreamt-wearable-sleep-staging/re...,None,
3,stage9_best,stage9,single_epoch_cnn,single_epoch_cnn,stage9_best_single_epoch_cnn,best validation macro F1 within stage/model class,0.384392,ready,/home/manns79/dreamt-wearable-sleep-staging/re...,None,
4,stage10_best,stage10,temporal_context_cnn,temporal_context_cnn,stage10_best,best validation macro F1 within stage/model class,0.378887,ready,/home/manns79/dreamt-wearable-sleep-staging/re...,None,
5,stage11_best,stage11,cnn_gru_many_to_one,cnn_gru_many_to_one,stage11_best_cnn_gru,best validation macro F1 within stage/model class,0.284394,ready,/home/manns79/dreamt-wearable-sleep-staging/re...,None,
6,stage12_best,stage12,cnn_gru_many_to_many,cnn_gru_many_to_many,stage12_best_cnn_gru_many_to_many,best validation macro F1 within stage/model class,0.366689,ready,/home/manns79/dreamt-wearable-sleep-staging/re...,None,
7,stage14_best,stage14,multiscale_residual_fusion,multiscale_residual_fusion,stage14_best_sqrt_weighted,best validation macro F1 within stage/model class,0.445788,ready,/home/manns79/dreamt-wearable-sleep-staging/re...,None,
8,stage15_equal_weight_seed_ensemble,stage15,frozen_stage14_embedding_tcn,frozen_stage14_embedding_tcn,stage15_equal_weight_seed_ensemble,equal-weight seed ensemble selected before tes...,0.491921,ready,/home/manns79/dreamt-wearable-sleep-staging/re...,None,
9,stage16_equal_weight_seed_ensemble,stage16,frozen_stage14_embedding_tcn_s61,frozen_stage14_embedding_tcn_s61,stage16_equal_weight_seed_ensemble,equal-weight seed ensemble selected before tes...,0.506251,ready,/home/manns79/dreamt-wearable-sleep-staging/re...,None,


All candidates are ready to freeze.


## Freeze Registry

Set `FREEZE_FINAL_CANDIDATE_REGISTRY = True` only after Stage 19 has completed and the displayed candidate list is the final pre-test list.

In [3]:
FREEZE_FINAL_CANDIDATE_REGISTRY = True

if FREEZE_FINAL_CANDIDATE_REGISTRY:
    paths = freeze_candidate_registry(registry, output_dir=stage20_output_dir)
    print("Frozen registry:", paths["registry"])
    print("Manifest:", paths["manifest"])
else:
    print("Final candidate registry not frozen in this run.")


Frozen registry: /home/manns79/dreamt-wearable-sleep-staging/results/stage20_final_test_evaluation/final_candidate_registry.csv
Manifest: /home/manns79/dreamt-wearable-sleep-staging/results/stage20_final_test_evaluation/final_candidate_registry_manifest.json


## Final Test Guard

The final evaluation cell materializes validation and guarded test prediction CSVs under `results/stage20_final_test_evaluation/predictions/` for the locked candidates before metric aggregation. Leave `RUN_FINAL_TEST_EVALUATION = False` until the registry is frozen and you are ready to evaluate the held-out test split once.

In [4]:
RUN_FINAL_TEST_EVALUATION = True

if RUN_FINAL_TEST_EVALUATION:
    require_final_test_confirmation(RUN_FINAL_TEST_EVALUATION)
    frozen_registry_path = stage20_output_dir / "final_candidate_registry.csv"
    frozen_registry = pd.read_csv(frozen_registry_path)
    assert_final_registry_ready(frozen_registry)

    materialization_manifest = materialize_final_candidate_predictions(
        frozen_registry,
        results_dir=repo_root / "results",
        output_dir=prediction_dir,
        run_final_test=True,
        overwrite=False,
    )
    display(materialization_manifest)

    validation_prediction_paths = sorted(prediction_dir.glob("validation_predictions_*.csv"))
    test_prediction_paths = sorted(prediction_dir.glob("test_predictions_*.csv"))
    validation_predictions = load_prediction_tables(validation_prediction_paths)
    test_predictions = load_prediction_tables(test_prediction_paths)

    outputs = run_final_test_evaluation(
        registry=frozen_registry,
        validation_predictions=validation_predictions,
        test_predictions=test_predictions,
        output_dir=stage20_output_dir,
        run_final_test=True,
        make_plots=True,
    )
    display(outputs["validation_test_metric_comparison"])
    display(outputs["duration_error_summary"])
else:
    print(FINAL_TEST_GUARD_MESSAGE)


,candidate_id,split,status,path,source_path
0,stage6_majority_class,validation,already_available,/home/manns79/dreamt-wearable-sleep-staging/re...,stage6_refit_from_train_features
1,stage6_logistic_regression,validation,already_available,/home/manns79/dreamt-wearable-sleep-staging/re...,stage6_refit_from_train_features
2,stage6_xgboost,validation,already_available,/home/manns79/dreamt-wearable-sleep-staging/re...,stage6_refit_from_train_features
3,stage9_best,validation,materialized,/home/manns79/dreamt-wearable-sleep-staging/re...,/home/manns79/dreamt-wearable-sleep-staging/re...
4,stage10_best,validation,materialized,/home/manns79/dreamt-wearable-sleep-staging/re...,/home/manns79/dreamt-wearable-sleep-staging/re...
5,stage11_best,validation,materialized,/home/manns79/dreamt-wearable-sleep-staging/re...,/home/manns79/dreamt-wearable-sleep-staging/re...
6,stage12_best,validation,materialized,/home/manns79/dreamt-wearable-sleep-staging/re...,/home/manns79/dreamt-wearable-sleep-staging/re...
7,stage14_best,validation,materialized,/home/manns79/dreamt-wearable-sleep-staging/re...,/home/manns79/dreamt-wearable-sleep-staging/re...
8,stage15_equal_weight_seed_ensemble,validation,materialized,/home/manns79/dreamt-wearable-sleep-staging/re...,/home/manns79/dreamt-wearable-sleep-staging/re...
9,stage16_equal_weight_seed_ensemble,validation,materialized,/home/manns79/dreamt-wearable-sleep-staging/re...,/home/manns79/dreamt-wearable-sleep-staging/re...


,stage,model_family,model_name,accuracy_validation,balanced_accuracy_validation,macro_f1_validation,Wake_precision_validation,Wake_recall_validation,Wake_f1_validation,Non_REM_precision_validation,...,macro_f1_test_minus_validation,Wake_precision_test_minus_validation,Wake_recall_test_minus_validation,Wake_f1_test_minus_validation,Non_REM_precision_test_minus_validation,Non_REM_recall_test_minus_validation,Non_REM_f1_test_minus_validation,REM_precision_test_minus_validation,REM_recall_test_minus_validation,REM_f1_test_minus_validation
0,stage10,temporal_context_cnn,stage10_best,0.569900,0.394959,0.378887,0.315789,0.446207,0.369838,0.728600,...,0.046142,0.064233,0.175762,0.101947,0.008457,-0.039754,-0.018017,0.117929,0.029492,0.054496
1,stage11,cnn_gru_many_to_one,stage11_best_cnn_gru,0.287303,0.392731,0.284394,0.261077,0.395652,0.314577,0.757095,...,0.044849,0.135815,0.045514,0.103283,-0.012310,0.067533,0.074937,-0.026890,-0.115856,-0.043672
2,stage12,cnn_gru_many_to_many,stage12_best_cnn_gru_many_to_many,0.466450,0.399966,0.366689,0.234915,0.466825,0.312550,0.752799,...,0.016594,0.116612,0.243711,0.157804,0.002844,-0.033035,-0.023402,-0.059889,-0.123414,-0.084621
3,stage14,multiscale_residual_fusion,stage14_best_sqrt_weighted,0.612372,0.452451,0.445788,0.368506,0.430332,0.397027,0.766613,...,0.007732,0.165035,0.054947,0.111240,-0.029870,0.090191,0.028645,-0.075612,-0.141382,-0.116690
4,stage15,frozen_stage14_embedding_tcn,stage15_equal_weight_seed_ensemble,0.642139,0.495485,0.491921,0.481164,0.478199,0.479677,0.772567,...,0.006391,0.134924,-0.000015,0.058769,-0.026211,0.134987,0.049006,0.062673,-0.160877,-0.088602
5,stage16,frozen_stage14_embedding_tcn_s61,stage16_equal_weight_seed_ensemble,0.661345,0.515297,0.506251,0.411196,0.563981,0.475620,0.784116,...,-0.006246,0.130239,0.022397,0.087391,-0.025906,0.070243,0.020794,0.019315,-0.132794,-0.126924
6,stage19,transition_regularized_frozen_stage14_tcn_s61,stage19_best_equal_weight_seed_ensemble_lambda...,0.663258,0.518628,0.509597,0.418214,0.565877,0.480967,0.785688,...,-0.008511,0.128271,0.015890,0.082607,-0.028563,0.073118,0.020683,0.031423,-0.139073,-0.128822
7,stage6,feature_baseline,logistic_elasticnet,0.402095,0.470829,0.374741,0.316752,0.467773,0.377727,0.796929,...,0.033184,0.165322,0.023536,0.108921,-0.063621,0.118416,0.084339,-0.056535,-0.253555,-0.093707
8,stage6,feature_baseline,xgboost_all_features,0.476594,0.429716,0.383188,0.292050,0.586730,0.389983,0.753680,...,0.051849,0.160375,0.018804,0.127918,-0.004998,0.082496,0.054576,-0.025527,-0.025451,-0.026946
9,stage6,sanity_baseline,majority_class,0.706411,0.333333,0.275983,0.000000,0.000000,0.000000,0.706411,...,-0.010244,0.000000,0.000000,0.000000,-0.043602,0.000000,-0.030733,0.000000,0.000000,0.000000


,stage,model_family,model_name,split,n_participants,mean_tst_error_minutes,mean_tst_absolute_error_minutes,median_tst_absolute_error_minutes,mean_rem_duration_error_minutes,mean_rem_duration_absolute_error_minutes,median_rem_duration_absolute_error_minutes
0,stage10,temporal_context_cnn,stage10_best,test,15,-57.766667,90.433333,36.0,-24.233333,46.366667,48.0
1,stage10,temporal_context_cnn,stage10_best,validation,15,-27.766667,72.366667,56.5,-11.000000,62.733333,40.5
2,stage11,cnn_gru_many_to_one,stage11_best_cnn_gru,test,15,-10.333333,43.200000,34.0,158.000000,160.333333,120.0
3,stage11,cnn_gru_many_to_one,stage11_best_cnn_gru,validation,15,-35.566667,69.166667,52.5,167.200000,180.666667,151.5
4,stage12,cnn_gru_many_to_many,stage12_best_cnn_gru_many_to_many,test,15,-95.966667,103.433333,79.5,5.266667,41.400000,33.0
5,stage12,cnn_gru_many_to_many,stage12_best_cnn_gru_many_to_many,validation,15,-69.433333,100.833333,67.5,23.233333,49.566667,37.5
6,stage14,multiscale_residual_fusion,stage14_best_sqrt_weighted,test,15,8.500000,44.033333,37.5,-20.900000,31.766667,15.5
7,stage14,multiscale_residual_fusion,stage14_best_sqrt_weighted,validation,15,-11.800000,38.066667,34.5,3.033333,32.633333,17.0
8,stage15,frozen_stage14_embedding_tcn,stage15_equal_weight_seed_ensemble,test,15,21.033333,45.700000,21.0,-28.033333,36.633333,37.0
9,stage15,frozen_stage14_embedding_tcn,stage15_equal_weight_seed_ensemble,validation,15,0.433333,41.633333,35.5,10.000000,51.333333,39.5
